# LifeMate — sửa metadata từ cache và benchmark tiếp nối

Notebook này **không chạy lại EDA audio, signal audit, hash hoặc tạo lại split**. Nó sử dụng CSV của lần chạy trước, sửa lỗi suy train/test do thư mục cha `train+public_test`, loại 3 dòng sample_submission bị đếm nhầm, kiểm tra lại validation/benchmark, rồi cho phép chạy baseline tùy chọn.

## Cách dùng
1. Tạo notebook mới, Add Input = **output notebook audit cũ**. Để Private.
2. Chạy với `RUN_BASELINES=False`: chỉ cần CSV, không cần mount audio gốc/GPU/Internet; tạo output đã sửa và ZIP.
3. Muốn benchmark: thêm dataset gốc, GPU và checkpoint hoặc Internet tải checkpoint nếu quy định cho phép; đặt `RUN_BASELINES=True` rồi chạy. Không có request gửi audio tới API ASR.
4. File audit cũ được giữ nguyên. Kết quả mới ở `/kaggle/working/lifemate_asr_followup`.

Không đọc `*_environment/` hoặc `state.json`. Đây là phân tích dữ liệu của đội đã đăng ký, không publish transcript/output. Không coi các tỷ lệ cờ regex là phân bố giọng hoặc code-switching đã xác minh.


In [ ]:
from pathlib import Path
from collections import Counter
import os, json, math, time, random, hashlib, unicodedata, gc, shutil
import numpy as np
import pandas as pd
from IPython.display import display, FileLink
SEED=2026
BASELINE_N=100
RUN_BASELINES=False
CACHE_DIR=None  # hoặc Path('/kaggle/input/TEN-OUTPUT/lifemate_asr_audit')
ROOT=Path('/kaggle/input/datasets/jamel27/zalo-aichallange-lifemate-dataset')
OUT=Path('/kaggle/working/lifemate_asr_followup')
OUT.mkdir(parents=True,exist_ok=True)
rng=np.random.default_rng(SEED)

def forbidden(p):
    p=Path(p)
    return p.name.lower()=='state.json' or any('_environment' in x.lower() or x.lower()=='environment' for x in p.parts)
def safe(p):
    p=Path(p)
    if forbidden(p): raise PermissionError('Environment/state blocked')
    q=p.resolve()
    if forbidden(q) or not q.is_relative_to(ROOT.resolve()): raise PermissionError('Outside ROOT or blocked symlink')
    return q

def save_csv(df,name): df.to_csv(OUT/name,index=False,encoding='utf-8-sig')
def normalize_text(text,punctuation=True):
    text=unicodedata.normalize('NFC',str(text or '')).casefold()
    if punctuation: text=''.join(' ' if unicodedata.category(c).startswith('P') else c for c in text)
    return ' '.join(text.split())
patterns={k:None for k in ['money_hint','time_hint','negation_hint','number_word_hint','location_hint','device_app_hint','english_hint','person_hint']}

if CACHE_DIR is None:
    hits=[]
    roots=[Path('/kaggle/working/lifemate_asr_audit'),Path('/kaggle/input')]
    for search_root in roots:
        if not search_root.exists(): continue
        for current,dirs,names in os.walk(search_root,followlinks=False):
            dirs[:]=[d for d in dirs if not forbidden(Path(current)/d) and not (Path(current)/d).is_symlink()]
            if 'references_enriched.csv' in names and not forbidden(current): hits.append(Path(current))
    hits=sorted(set(hits))
    if len(hits)!=1: raise RuntimeError(f'Tìm thấy {len(hits)} output audit. Đặt CACHE_DIR tường minh: {hits}')
    CACHE_DIR=hits[0]
else: CACHE_DIR=Path(CACHE_DIR)
assert CACHE_DIR.is_dir() and not forbidden(CACHE_DIR)
print('Cached audit:',CACHE_DIR)


## 1. Sửa split bằng đúng tên manifest, không dùng token trong thư mục cha
Những tên ngoài `train.jsonl` và `public_test.jsonl` không được đoán. `sample_submission.jsonl` không phải sample audio, nên bị loại khỏi sample counts.


In [ ]:
def split_from_manifest(value):
    name=Path(str(value).replace('\\','/')).name.lower()
    if name in {'train.jsonl','train.json'}: return 'train'
    if name in {'public_test.jsonl','public_test.json','test.jsonl','test.json'}: return 'test'
    raise ValueError(f'Manifest chưa được duyệt: {name}')
def read_cached(name):
    p=CACHE_DIR/name
    if forbidden(p): raise PermissionError('Blocked')
    return pd.read_csv(p)

refs=read_cached('references_enriched.csv')
refs['split_original']=refs['split']
refs['split']=refs.manifest.map(split_from_manifest)
assert refs.ref_uid.is_unique
sample_stats=read_cached('sample_coverage.csv')
sample_stats['manifest_from_uid']=sample_stats.sample_uid.str.rsplit(':',n=1).str[0]
excluded=sample_stats[sample_stats.manifest_from_uid.map(lambda x:Path(x).name.lower()).eq('sample_submission.jsonl')].copy()
sample_stats=sample_stats.drop(excluded.index).copy()
sample_stats['split']=sample_stats.manifest_from_uid.map(split_from_manifest)
assert not sample_stats.final_audio_refs.eq(0).any(), 'Vẫn có sample thiếu final audio: cần kiểm tra.'
audio_df=read_cached('audio_audit.csv')
mapping=refs.groupby('path').split.agg(lambda x:'+'.join(sorted(set(x))))
audio_df['ref_split']=audio_df.path.map(mapping).fillna('unreferenced')
# Các CSV có split khác được sửa cùng cách; không thay đổi ref_uid hoặc đường dẫn audio.
for name in ['audio_references.csv','manual_annotations.csv']:
    d=read_cached(name)
    d['split']=d.ref_uid.map(refs.set_index('ref_uid').split)
    assert d.split.notna().all()
    save_csv(d,name)
workload=read_cached('sample_audio_workload.csv')
workload['split']=workload.sample_uid.map(sample_stats.set_index('sample_uid').split)
assert workload.split.notna().all()
save_csv(workload,'sample_audio_workload.csv')
save_csv(refs,'references_enriched.csv'); save_csv(audio_df,'audio_audit.csv')
save_csv(sample_stats,'sample_coverage.csv'); save_csv(excluded,'excluded_submission_rows.csv')
headers=read_cached('audio_headers.csv')
headers['ref_split']=headers.path.map(mapping).fillna('unreferenced')
save_csv(headers,'audio_headers.csv')
for name in ['near_text_pairs.csv','schema_paths.csv','parse_errors.csv','train_token_frequencies.csv']:
    if (CACHE_DIR/name).is_file(): shutil.copyfile(CACHE_DIR/name,OUT/name)
print('Excluded sample_submission rows:',len(excluded))
display(sample_stats.groupby('split').agg(samples=('sample_uid','size')))
display(refs.groupby(['split','kind']).agg(audio=('ref_uid','size'),references=('has_reference','sum'),hours=('duration',lambda s:s.sum()/3600)))


## 2. Kiểm tra split đã lưu trước khi giữ lại
Dù nhãn split cũ bị sai, bước tạo validation trước đó đã lọc có transcript, nên tập được lưu có thể vẫn chỉ chứa train thật. Notebook kiểm chứng điều đó thay vì tạo lại split. Không có speaker ID -> không khẳng định speaker-disjoint.


In [ ]:
split_manifest=read_cached('split_manifest.csv')
assert split_manifest.ref_uid.is_unique
joined=split_manifest[['ref_uid','group_id','partition']].merge(refs,on='ref_uid',validate='one_to_one')
assert len(joined)==len(split_manifest)
assert joined.split.eq('train').all(), 'Split cũ có test: dừng, cần tạo lại split từ metadata.'
assert joined.has_reference.all()
validation=joined[joined.partition=='validation'].copy()
development=joined[joined.partition=='development'].copy()
assert len(validation) and len(development)
for col in ['sample_uid','group_id','path','file_sha256','decoded_sha256','reference_norm']:
    assert not(set(validation[col].dropna())&set(development[col].dropna())), f'Leakage development/validation: {col}'
overlap=[]
test=refs[refs.split=='test']
for col in ['path','file_sha256','decoded_sha256']:
    hits=joined[joined[col].isin(set(test[col].dropna()))]
    for _,r in hits.iterrows(): overlap.append({'ref_uid':r.ref_uid,'overlap_type':col})
assert not overlap, f'Có train/test overlap cần giải quyết: {len(overlap)}'
benchmark_old=read_cached('benchmark_manifest.csv')
assert set(benchmark_old.ref_uid).issubset(set(validation.ref_uid))
assert refs.set_index('ref_uid').loc[benchmark_old.ref_uid,'split'].eq('train').all()
save_csv(split_manifest,'split_manifest.csv')
save_csv(pd.DataFrame(overlap,columns=['ref_uid','overlap_type']),'train_test_overlap.csv')
display(joined.groupby('partition').agg(audio=('ref_uid','size'),samples=('sample_uid','nunique'),groups=('group_id','nunique')))
print('Benchmark cũ hợp lệ:',len(benchmark_old),'audio; nhãn split sai KHÔNG đưa test vào tập có đáp án này.')
print('Không có exact file-hash overlap train/test; decoded hash chỉ bao phủ subset đã audit.')


## 3. Benchmark tùy chọn
Mặc định tắt. Chạy EDA cache không cần SoundFile/torch/transformers. Khi bật, cần mount dataset gốc đúng `ROOT`, cài dependencies, chuẩn bị GPU/checkpoints theo quy định.

Harness: batch=1, greedy, không VAD/denoise/context bias; Whisper language=vi, task=transcribe; file <=30s, không cắt audio ngầm. Hai baseline đầu là checkpoint Whisper; baseline thứ ba là CTC. Không có kết quả thực nghiệm nào được tạo sẵn.

Nguồn checkpoint:
- https://huggingface.co/openai/whisper-small
- https://huggingface.co/vinai/PhoWhisper-small
- https://huggingface.co/nguyenvulebinh/wav2vec2-base-vietnamese-250h

Trước khi chạy 3 model, nên smoke test một model với `BASELINE_N=10`; sau đó dùng cùng tập 100 audio để so sánh. Để chốt, tăng lên toàn validation với `BASELINE_N=len(validation)`. Tập chọn theo strata không đại diện tỷ lệ final/history thực tế; báo riêng từng loại.


In [ ]:
CANDIDATES=[
    {'name':'whisper_small','model_id':'openai/whisper-small','family':'seq2seq','revision':'main'},
    {'name':'phowhisper_small','model_id':'vinai/PhoWhisper-small','family':'seq2seq','revision':'main'},
    {'name':'wav2vec2_vi','model_id':'nguyenvulebinh/wav2vec2-base-vietnamese-250h','family':'ctc','revision':'main'},
]
# Chỉ sửa trước khi so sánh. Sau khi chốt: lưu revision commit cụ thể thay cho main.
bench_pool=validation[validation.duration.gt(0)&validation.duration.le(30)].copy()
bench_pool=bench_pool.drop_duplicates(['path','reference_norm'])
# Stratify round-robin theo final/history, độ dài, nhóm nội dung quan trọng.
bench_pool['bench_bin']=pd.cut(bench_pool.duration,[0,3,10,30],include_lowest=True).astype(str)
bench_pool['critical_hint']=bench_pool[['money_hint','time_hint','negation_hint','person_hint']].any(axis=1)
bs=[list(g.sample(frac=1,random_state=SEED).index) for _,g in
    bench_pool.groupby(['kind','bench_bin','critical_hint'],observed=True)]
chosen=[]
while any(bs) and len(chosen)<BASELINE_N:
    for b in bs:
        if b and len(chosen)<BASELINE_N: chosen.append(b.pop())
benchmark=bench_pool.loc[chosen].copy()
save_csv(benchmark,'benchmark_manifest.csv')
print('Validation refs:',len(validation),'| eligible <=30s unique pairs:',len(bench_pool),'| benchmark:',len(benchmark))
print('Không chấm các file >30s bằng harness này; cần bài đo long-form riêng.')
display(benchmark.groupby('kind').size().rename('benchmark_refs'))


In [ ]:
def resample_mono(path,target_sr):
    x,sr=sf.read(str(safe(path)),dtype='float32',always_2d=True)
    x=x.mean(axis=1)
    if sr!=target_sr:
        g=math.gcd(int(sr),int(target_sr))
        x=signal.resample_poly(x,target_sr//g,sr//g).astype('float32')
    return np.ascontiguousarray(x,dtype='float32')

benchmark_predictions=[]; model_runs=[]
if RUN_BASELINES:
    import soundfile as sf
    from scipy import signal
    assert ROOT.is_dir(), 'Add Input dataset gốc trước khi benchmark.'
    assert len(benchmark), 'Validation/benchmark rỗng; giải quyết split trước.'
    import torch, psutil, threading
    from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, AutoModelForCTC
    device='cuda' if torch.cuda.is_available() else 'cpu'
    process=psutil.Process()
    def sync():
        if device=='cuda': torch.cuda.synchronize()
    for spec in CANDIDATES:
        print('Running:',spec['name'],'on',device)
        model=processor=None
        started=time.perf_counter()
        try:
            processor=AutoProcessor.from_pretrained(spec['model_id'],revision=spec['revision'],trust_remote_code=False)
            cls=AutoModelForSpeechSeq2Seq if spec['family']=='seq2seq' else AutoModelForCTC
            model=cls.from_pretrained(spec['model_id'],revision=spec['revision'],trust_remote_code=False)
            model=model.to(device).eval()
            target_sr=int(processor.feature_extractor.sampling_rate)
            load_seconds=time.perf_counter()-started
            resolved_revision=getattr(model.config,'_commit_hash',None)
            def forward_decode(x):
                batch=processor(x,sampling_rate=target_sr,return_tensors='pt')
                batch={k:v.to(device) for k,v in batch.items()}
                with torch.inference_mode():
                    if spec['family']=='seq2seq':
                        # Xóa forced_decoder_ids legacy; language/task được đặt tường minh.
                        if hasattr(model.generation_config,'forced_decoder_ids'): model.generation_config.forced_decoder_ids=None
                        ids=model.generate(**batch,language='vi',task='transcribe',num_beams=1,do_sample=False,max_new_tokens=440)
                        saturated=bool(ids.shape[-1]>=440)
                    else:
                        ids=model(**batch).logits.argmax(dim=-1); saturated=False
                return processor.batch_decode(ids)[0],saturated
            # Warm-up một input cố định; không đo vào steady-state latency.
            warm=resample_mono(benchmark.iloc[0].path,target_sr)
            forward_decode(warm); sync()
            for _,r in benchmark.iterrows():
                if device=='cuda': torch.cuda.reset_peak_memory_stats()
                peak=[process.memory_info().rss]; stop=threading.Event()
                def monitor():
                    while not stop.wait(.02): peak[0]=max(peak[0],process.memory_info().rss)
                watcher=threading.Thread(target=monitor,daemon=True); watcher.start()
                sync(); t0=time.perf_counter()
                status='ok'; error=''; hyp=''; saturated=False; compute_seconds=np.nan
                try:
                    x=resample_mono(r.path,target_sr)
                    sync(); t1=time.perf_counter()
                    hyp,saturated=forward_decode(x)
                    sync(); compute_seconds=time.perf_counter()-t1
                except Exception as e:
                    status='error'; error=f'{type(e).__name__}: {e}'[:300]
                    if device=='cuda': torch.cuda.empty_cache()
                finally:
                    sync(); e2e=time.perf_counter()-t0
                    peak[0]=max(peak[0],process.memory_info().rss); stop.set(); watcher.join()
                benchmark_predictions.append({'model':spec['name'],'ref_uid':r.ref_uid,'hypothesis':hyp,
                    'status':status,'error':error,'e2e_seconds':e2e,'compute_seconds':compute_seconds,
                    'rtf':e2e/r.duration,'rss_peak_sampled_mb':peak[0]/2**20,
                    'cuda_peak_allocated_mb':torch.cuda.max_memory_allocated()/2**20 if device=='cuda' else np.nan,
                    'generation_limit_flag':saturated})
            model_runs.append(spec|{'status':'ok','device':device,'target_sr':target_sr,'load_seconds':load_seconds,
                                    'resolved_revision':resolved_revision,'dtype':str(next(model.parameters()).dtype)})
        except Exception as e:
            model_runs.append(spec|{'status':'load_or_warmup_error','error':f'{type(e).__name__}: {e}'[:400]})
        finally:
            del model,processor
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        save_csv(pd.DataFrame(benchmark_predictions),'predictions.csv')
        (OUT/'model_runs.json').write_text(json.dumps(model_runs,ensure_ascii=False,indent=2),encoding='utf-8')
else:
    print('Benchmark tắt. Bật RUN_BASELINES sau khi kiểm tra schema, luật và validation.')


### Đọc chỉ số
WER/CER corpus: tổng lỗi chia tổng độ dài reference, giữ dấu tiếng Việt. Có hai chế độ normalization; không tự đổi số chữ sang chữ số. CER bỏ khoảng trắng. Failure được giữ và chấm như hypothesis rỗng; load error không có điểm giả. Entity-hint slice chỉ là WER trên câu có cờ, không phải entity error rate.

E2E latency gồm đọc/resample/processor/model/decoding, không gồm tải model và warm-up. RTF=e2e/audio duration. RAM là RSS toàn process lấy mẫu 20ms; VRAM là peak allocated PyTorch, không phải toàn GPU. Benchmark một lần/file, không phải kiểm tra độ ổn định dài hạn. Bootstrap theo group_id, không theo token.


In [ ]:
def edit_counts(ref_tokens,hyp_tokens):
    # DP Levenshtein có đếm S,D,I; tie-break cố định substitution > deletion > insertion.
    previous=[(j,0,0,j) for j in range(len(hyp_tokens)+1)]
    for i,a in enumerate(ref_tokens,1):
        current=[(i,0,i,0)]
        for j,b in enumerate(hyp_tokens,1):
            if a==b: current.append(previous[j-1]); continue
            cost,s,d,ins=previous[j-1]; sub=(cost+1,s+1,d,ins)
            cost,s,d,ins=previous[j]; delete=(cost+1,s,d+1,ins)
            cost,s,d,ins=current[j-1]; insert=(cost+1,s,d,ins+1)
            current.append(min([sub,delete,insert],key=lambda z:z[0]))
        previous=current
    _,s,d,i=previous[-1]
    return s,d,i

def score_pair(ref,hyp,punctuation):
    r=normalize_text(ref,punctuation); h=normalize_text(hyp,punctuation)
    s,d,i=edit_counts(r.split(),h.split())
    cr=list(r.replace(' ','')); ch=list(h.replace(' ',''))
    cs,cd,ci=edit_counts(cr,ch)
    return {'S':s,'D':d,'I':i,'N':len(r.split()),'char_errors':cs+cd+ci,'char_N':len(cr),
            'exact_match':r==h}

predictions=pd.DataFrame(benchmark_predictions)
scored=pd.DataFrame(); leaderboard=pd.DataFrame(); slices=pd.DataFrame()
if len(predictions):
    scored=predictions.merge(benchmark,on='ref_uid',validate='many_to_one')
    metric_rows=[]
    for _,r in scored.iterrows():
        for mode,punct in [('keep_punctuation',False),('light_normalized',True)]:
            metric_rows.append(r.to_dict()|{'normalization':mode}|score_pair(r.reference,r.hypothesis,punct))
    scored=pd.DataFrame(metric_rows)
    def summarize(g):
        n=g.N.sum(); cn=g.char_N.sum()
        return pd.Series({'n':len(g),'WER':(g.S.sum()+g.D.sum()+g.I.sum())/n if n else np.nan,
            'CER':g.char_errors.sum()/cn if cn else np.nan,'exact_match':g.exact_match.mean(),
            'failure_rate':g.status.ne('ok').mean(),'latency_p50_s':g.e2e_seconds.median(),
            'latency_p95_s':g.e2e_seconds.quantile(.95),'RTF_total':g.e2e_seconds.sum()/g.duration.sum(),
            'max_sampled_RSS_MB':g.rss_peak_sampled_mb.max(),'max_VRAM_alloc_MB':g.cuda_peak_allocated_mb.max()})
    leaderboard=scored.groupby(['model','normalization']).apply(summarize)
    display(leaderboard)
    light=scored[scored.normalization=='light_normalized'].copy()
    slice_rows=[]
    for model,g in light.groupby('model'):
        for name in ['all','final','history']+list(patterns)+['flag_low_level','flag_near_fullscale']:
            sub=g if name=='all' else g[g.kind==name] if name in {'final','history'} else g[g[name].fillna(False)]
            if len(sub): slice_rows.append({'model':model,'slice':name}|summarize(sub).to_dict())
    slices=pd.DataFrame(slice_rows); display(slices)
    save_csv(scored,'scored_predictions.csv'); save_csv(leaderboard.reset_index(),'leaderboard.csv'); save_csv(slices,'slice_metrics.csv')
    print('Slice có entity-hint là WER trên câu có cờ; KHÔNG phải entity error rate.')
else:
    print('Chưa có predictions; phần EDA vẫn hoàn tất. Không tạo bảng xếp hạng giả.')


In [ ]:
def paired_bootstrap_wer(scored_df, model_a, model_b, repeats=1000):
    light=scored_df[scored_df.normalization=='light_normalized'].copy()
    light['errors']=light.S+light.D+light.I
    a=light[light.model==model_a].set_index('ref_uid')
    b=light[light.model==model_b].set_index('ref_uid')
    if set(a.index)!=set(b.index) or not len(a):
        raise ValueError('Hai model phải có predictions trên đúng cùng tập ref_uid, bao gồm failure.')
    joined=a[['group_id','errors','N']].rename(columns={'errors':'errors_a'}).join(b[['errors']].rename(columns={'errors':'errors_b'}))
    agg=joined.groupby('group_id')[['errors_a','errors_b','N']].sum().to_numpy()
    if len(agg)<2: return {'independent_groups':len(agg),'warning':'Không đủ nhóm để bootstrap'}
    local_rng=np.random.default_rng(SEED)
    deltas=[]
    for _ in range(repeats):
        z=agg[local_rng.integers(0,len(agg),len(agg))].sum(axis=0)
        deltas.append((z[1]-z[0])/z[2])
    return {'A':model_a,'B':model_b,'independent_groups':len(agg),
            'delta_WER_B_minus_A':float((agg[:,1].sum()-agg[:,0].sum())/agg[:,2].sum()),
            'percentile_95_low':float(np.quantile(deltas,.025)),
            'percentile_95_high':float(np.quantile(deltas,.975)),
            'warning':'Ít nhóm; diễn giải thận trọng' if len(agg)<30 else ''}

bootstrap_rows=[]
if len(scored):
    names=list(scored.model.unique())
    for other in names[1:]: bootstrap_rows.append(paired_bootstrap_wer(scored,names[0],other))
    display(pd.DataFrame(bootstrap_rows))
save_csv(pd.DataFrame(bootstrap_rows),'paired_bootstrap.csv')


In [ ]:
gold=benchmark[['ref_uid','reference']].copy()
for c in ['person_spans_json','money_spans_json','time_spans_json','location_spans_json','device_spans_json','negation_spans_json']:
    gold[c]='[]'
save_csv(gold,'critical_gold_annotation.csv')
if len(predictions):
    judgments=predictions[['model','ref_uid','hypothesis','status']].merge(benchmark[['ref_uid','reference']],on='ref_uid')
else:
    judgments=pd.DataFrame(columns=['model','ref_uid','hypothesis','status','reference'])
for c in ['person_judgment','money_judgment','time_judgment','location_judgment','device_judgment','negation_judgment','notes']:
    judgments[c]=''
save_csv(judgments,'critical_model_review.csv')

def summarize_manual_judgments(path):
    df=pd.read_csv(path).fillna('')
    result=[]
    for model,g in df.groupby('model'):
        for c in [x for x in g if x.endswith('_judgment')]:
            counts=g[c].value_counts(); n=int(counts.get('correct',0)+counts.get('incorrect',0))
            result.append({'model':model,'entity':c,'adjudicated':n,
                'error_rate':counts.get('incorrect',0)/n if n else np.nan,
                'uncertain':int(counts.get('uncertain',0)),'unfilled':int(counts.get('',0))})
    return pd.DataFrame(result)


## 4. Xuất kết quả đã sửa và ZIP
Chạy cả khi `RUN_BASELINES=False`; không cần đọc lại audio. ZIP có transcript nên chỉ dùng trong phạm vi đội được phép. Mẫu nhãn thủ công được tạo lại cho benchmark mới, không ghi đè file đã annotate ở CACHE_DIR.


In [ ]:
summary={
 'samples_train':int(sample_stats.split.eq('train').sum()),
 'samples_public_test':int(sample_stats.split.eq('test').sum()),
 'excluded_submission_rows':len(excluded),'audio_files':len(audio_df),
 'resolved_refs':int(refs.path.notna().sum()),'validation_refs':len(validation),'benchmark_refs':len(benchmark),
 'signal_files_previously_audited':int(audio_df.signal_status.eq('ok').sum()),
 'run_baselines':RUN_BASELINES,
 'notes':['Split labels repaired by exact manifest filename.',
          'Original validation verified train-only and preserved.',
          'No new audio scan/hash; signal coverage remains 2000 files in this uploaded run.',
          'No speaker IDs; accents/noise/code-switching not validated by EDA.',
          'No state/environment reads; no external ASR API.']}
(OUT/'audit_summary_corrected.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
(OUT/'benchmark_config.json').write_text(json.dumps({'candidates':CANDIDATES,'seed':SEED,'baseline_n':BASELINE_N,
        'run_baselines':RUN_BASELINES},ensure_ascii=False,indent=2),encoding='utf-8')
zip_path=shutil.make_archive(str(OUT),'zip',root_dir=OUT.parent,base_dir=OUT.name)
print('Metadata đã sửa; không chạy lại EDA audio. ZIP:',zip_path)
display(FileLink(zip_path))
